# Tool-Router Ladder: LoRA SFT (V1-09)

**Settings:** Accelerator **GPU T4**, Internet **on**, and the private dataset attached (Add Input). The dataset holds `train.jsonl`, `manifest.json` and the `prompts-*.jsonl` files.

**Secrets:** `HF_TOKEN` (write) and `WANDB_API_KEY`, both ticked for this notebook.

**Do not install vLLM in this notebook.** It replaces torch. Generation has its own notebook.

Order of cells:
1. Memory check: one forward and backward on the longest episode, proving the 12,176-token worst case fits in T4 memory.
2. 0.5B.
3. 1.5B.

Each run checkpoints every epoch to the Hub and resumes if the session dies.

**Send back:** the two Hub repo ids and the last `[train] done: ...` line of each run.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "WANDB_API_KEY"])

In [ ]:
# Fresh clone every session: a relative clone + relative %cd nests the repo.
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

!pip install -q peft
# Kaggle ships torchao 0.10.0; peft's LoRA dispatcher raises on anything below
# 0.16.0 instead of skipping it. Nothing here uses torchao, and with the package
# gone peft's is_torchao_available() returns False and dispatch falls through.
!pip uninstall -y -q torchao

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

### 1. T4 memory check: about 2 minutes, no dataset needed.

`--smoke` is unavailable until `manifest.smoke.json` is rebuilt — the copy in the dataset
predates the 2026-09-22 prompt-template change, and a smoke run carries its own teacher
fingerprint, so it cannot fall back to the full manifest.

This runs what the smoke run existed to prove: one forward and backward at the worst-case
12,176-token episode, through the real `load_base_model`, `wrap_lora` and `masked_lm_loss`.
Measured on a T4 on 2026-09-23: **5.43 GiB** at 0.5B and **8.38 GiB** at 1.5B, of 14.56 GiB.

If either line OOMs, stop and send the error — do not start the real runs.

In [ ]:
import gc, torch
from pathlib import Path
from src.config import load_config
from src.train import load_base_model, masked_lm_loss, wrap_lora

L, N_TARGET = 12176, 1300   # longest episode; 137,149 target tokens / 108 episodes

def check(name):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    cfg = load_config(Path(f"configs/{name}.yaml"))
    dev = torch.device("cuda")
    model = wrap_lora(cfg, load_base_model(cfg, dev)); model.train()
    ids = torch.randint(0, 150000, (1, L), device=dev)
    labels = torch.full((1, L), -100, device=dev)
    labels[:, -N_TARGET:] = ids[:, -N_TARGET:]
    with torch.autocast("cuda", dtype=torch.float16):
        loss_sum, n = masked_lm_loss(model, ids, labels)
    (loss_sum / n).backward()
    print(f"  {cfg.model.base_model:<32} peak {torch.cuda.max_memory_allocated() / 2**30:5.2f} GiB")
    del model; gc.collect(); torch.cuda.empty_cache()

print(f"one episode at {L:,} tokens, of 14.56 GiB:")
check("qwen05b")
check("qwen15b")

### 2. Qwen2.5-0.5B: the first real run tells us the time; about 21 optimizer steps.

In [ ]:
!python src/train.py --config configs/qwen05b.yaml --data "$DATA" --output-dir /kaggle/working/qwen05b --hub-repo "$HF_USER/tool-router-qwen05b-sft" --wandb

### 3. Qwen2.5-1.5B

In [ ]:
!python src/train.py --config configs/qwen15b.yaml --data "$DATA" --output-dir /kaggle/working/qwen15b --hub-repo "$HF_USER/tool-router-qwen15b-sft" --wandb